In [ ]:
!pip -q install -U transformers datasets accelerate bitsandbytes sentencepiece sentence_transformers faiss-cpu evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 129.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.0 MB/s eta 0:00:00


In [ ]:
import re
import json
import random
import numpy as np
import torch
import faiss
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

### LLM-as-Judge  preference-пар для DPO

In [ ]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

policy_id = "Qwen/Qwen2.5-0.5B-Instruct"

policy_tokenizer = AutoTokenizer.from_pretrained(policy_id, use_fast=True)
policy = AutoModelForCausalLM.from_pretrained(policy_id, quantization_config=bnb, device_map="auto")
print(f"Loaded policy: {policy_id}")


judge_id = "Qwen/Qwen2.5-1.5B-Instruct"

judge_tokenizer = AutoTokenizer.from_pretrained(judge_id, use_fast=True)
judge = AutoModelForCausalLM.from_pretrained(judge_id, quantization_config=bnb, device_map="auto")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded policy: Qwen/Qwen2.5-0.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
def _build_chat_input(tokenizer, prompt, system_prompt="Ты полезный русскоязычный ассистент. Пиши естественно, грамотно и по делу."):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return tokenizer(text, return_tensors="pt")

In [ ]:
def decode_new_tokens(tokenizer, input_ids, output_ids):
    new_ids = output_ids[0, input_ids.shape[-1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

In [ ]:
def generate_samples(model, tokenizer, prompt, n=2, temperatures=(0.9, 0.7), top_p=0.9, max_new=220):
    inp = _build_chat_input(tokenizer, prompt)
    inp = {k: v.to(model.device) for k, v in inp.items()}
    outs = []
    with torch.no_grad():
        for i in range(n):
            temp = temperatures[i % len(temperatures)]
            out_ids = model.generate(
                **inp,
                do_sample=True,
                temperature=temp,
                top_p=top_p,
                max_new_tokens=max_new,
                repetition_penalty=1.08,
                no_repeat_ngram_size=3,
            )
            outs.append(decode_new_tokens(tokenizer, inp["input_ids"], out_ids))
    return outs

In [ ]:
def gen_two(policy, tokenizer, prompt, T=0.9, top_p=0.9, max_new=200):
    samples = generate_samples(
        policy,
        tokenizer,
        prompt,
        n=2,
        temperatures=(T, max(0.6, T - 0.2)),
        top_p=top_p,
        max_new=max_new,
    )
    return samples[0], samples[1]

In [ ]:
def judge_once(prompt, ans_a, ans_b):
    rubric = (
        "Ты строгий русскоязычный оценщик. Приоритеты в порядке важности: "
        "(1) качество русского языка: грамотность, естественность, стиль; "
        "(2) фактическая корректность; "
        "(3) безопасность; "
        "(4) следование инструкции. "
        "Верни СТРОГО JSON: {\"choice\":\"A|B\",\"confidence\":0.0-1.0,\"reason\":\"кратко\"}."
    )
    jp = (
        f"{rubric}\n"
        f"Запрос:\n{prompt}\n\n"
        f"Ответ A:\n{ans_a}\n\n"
        f"Ответ B:\n{ans_b}\n"
    )
    inp = _build_chat_input(judge_tokenizer, jp, system_prompt="Ты надежный русскоязычный LLM-судья.")
    inp = {k: v.to(judge.device) for k, v in inp.items()}

    with torch.no_grad():
        out_ids = judge.generate(
            **inp,
            do_sample=False,
            max_new_tokens=120,
        )
    txt = decode_new_tokens(judge_tokenizer, inp["input_ids"], out_ids)

    match = re.search(r"\{.*\}", txt, flags=re.S)
    if not match:
        return None, 0.5, txt

    try:
        parsed = json.loads(match.group(0))
    except Exception:
        return None, 0.5, txt

    choice = str(parsed.get("choice", "")).strip().upper()
    conf = float(parsed.get("confidence", 0.5))
    conf = max(0.0, min(1.0, conf))
    if choice not in {"A", "B"}:
        choice = None
    return choice, conf, txt

In [ ]:
def judge_pick(prompt, a, b):
    c1, s1, t1 = judge_once(prompt, a, b)
    c2, s2, t2 = judge_once(prompt, b, a)

    votes_for_a = 0.0
    if c1 == "A":
        votes_for_a += s1
    elif c1 == "B":
        votes_for_a += (1.0 - s1)

    if c2 == "B":
        votes_for_a += s2
    elif c2 == "A":
        votes_for_a += (1.0 - s2)

    conf = votes_for_a / 2.0
    choice = "A" if conf >= 0.5 else "B"
    return choice, abs(conf - 0.5) * 2.0, {"forward": t1, "swap": t2}

In [ ]:
prompts = [
    "Объясни в двух предложениях разницу между DPO и PPO в контексте RLHF.",
    "Что такое over-refusal в LLM-системах и как его снизить (3 пункта).",
    "Дай 5 практик безопасной разработки на Python для production-сервиса.",
    "Назови лучшие практики написания поддерживаемого кода на Python.",
    "Как писать тесты для ML-моделей в продакшене?",
]

pairs = []
for p in prompts:
    a, b = gen_two(policy, policy_tokenizer, p)
    choice, conf, raw = judge_pick(p, a, b)
    if choice and conf >= 0.55 and a[:30] != b[:30]:
        pairs.append({
            "prompt": p,
            "chosen": a if choice == "A" else b,
            "rejected": b if choice == "A" else a,
            "confidence": round(conf, 3),
        })

print(f"Built {len(pairs)} DPO pairs")
pairs[:2]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Built 2 DPO pairs


[{'prompt': 'Дай 5 практик безопасной разработки на Python для production-сервиса.',
  'chosen': 'Конечно! Мы будем заниматься этим с точки зрения Pythonа на простоту и надежности. Вот 5 наиболее подходящих практик для разработкой на Python в производительных средах:\n\n1. **Установка библиотек:**  \n   - Установите нужную вам Python библеотеку (например, `numpy`, `pandas`, `requests`) на компьютере.\n   \n2. **Загрузка кода в Docker:**  \n  - Используйте команду `docker run` для загрузки вашего скрипта в Docker образ (например `python3:3.8`).\n\n3. **Создание проекта на Git:**  \n- Важно создать папку с вашим проектом и установить `pip` (или `pip3`, если у вас есть более старые версии) в папке `/src`.\n\n4.',
  'rejected': 'Конечно! Практика безопасности - ключ к устойчивой работе любой организации. В данном разделе я покажу 5 важных шагов на Python, которые помогут вам безопасно развивать ваш проект на production-сервисе.\n\n### 1. Сосредоточься на надежности данных\n\n**Пример:** \n

### Best-of-N + RM-reranking

In [ ]:
rm_id = "OpenAssistant/reward-model-deberta-v3-large-v2"
rm_tokenizer = AutoTokenizer.from_pretrained(rm_id, use_fast=True)
rm = AutoModelForSequenceClassification.from_pretrained(rm_id).to("cuda")

config.json:   0%|          | 0.00/993 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/455 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: OpenAssistant/reward-model-deberta-v3-large-v2
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

In [ ]:
def rm_score(prompt, responses):
    texts = [f"Human: {prompt}\nAssistant: {r}" for r in responses]
    inputs = rm_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to("cuda")
    with torch.no_grad():
        s = rm(**inputs).logits.squeeze(-1).float()
    return s

In [ ]:
def length_penalty(text, target_len=900):
    # Мягкий штраф за слишком длинные ответы.
    over = max(0, len(text) - target_len)
    return 0.0005 * over

In [ ]:
def best_of_n(prompt, N=8):
    cands = []
    for _ in range(N):
        c = generate_samples(policy, policy_tokenizer, prompt, n=1, temperatures=(0.95, 0.85, 0.75), max_new=240)[0]
        cands.append(c)

    raw_scores = rm_score(prompt, cands)
    z = (raw_scores - raw_scores.mean()) / (raw_scores.std() + 1e-6)
    final_scores = []
    for c, s in zip(cands, z.tolist()):
        final_scores.append(s - length_penalty(c))

    idx = int(np.argmax(final_scores))
    ranked = sorted(zip(cands, final_scores), key=lambda x: x[1], reverse=True)
    return cands[idx], ranked

In [ ]:
p = "List 6 steps to implement PPO for RLHF."
best, scored = best_of_n(p, N=6)
print("BEST:", best[:300])

BEST: Sure, here are six steps you can follow to implement Policy Gradient Optimization (PPO) for reinforcement learning (RLHF):

1. **Define the Environment**: Clearly define your environment or problem space. This includes specifying what actions each state corresponds to and how the environment behaves


In [ ]:
scored

[('Sure, here are six steps you can follow to implement Policy Gradient Optimization (PPO) for reinforcement learning (RLHF):\n\n1. **Define the Environment**: Clearly define your environment or problem space. This includes specifying what actions each state corresponds to and how the environment behaves.\n\n2. **Create a Model**: Choose a model architecture that is suitable for your application. Common choices include Linear-Activation networks, Convolutional Neural Networks, Recurrent neural networks, etc.\n\n3. **Prepare the Training Data**: Prepare data sets containing interactions between the model and the environment over time. Each interaction should be a sequence of observations, actions, rewards, etc., and labels indicating whether an action was chosen or not.\n\n4. **Train the Model**: Use techniques like gradient ascent (SGD), mini-batch optimization, and momentum-based methods. Implement these in an optimizer like Adam, RMSprop, etc. You may also want to experiment with oth

### Constitutional AI - Конституционное переписывание

In [ ]:
CONSTITUTION = [
    "Не раскрывай персональные данные (PII) и не помогай с деанонимизацией.",
    "Если не уверен в фактах, прямо укажи неуверенность или попроси источник.",
    "По возможности предлагай безопасную и полезную альтернативу вместо сухого отказа.",
    "Пиши на грамотном, естественном русском языке без канцелярита.",
]

def constitutional_generate(prompt, critique_rounds=1):
    answer = gen_two(policy, policy_tokenizer, prompt)[0]
    print(f'answer 1: {answer}')

    for _ in range(critique_rounds):
        critique_prompt = (
            "Проверь ответ по правилам и верни строгий JSON с ключами "
            "violations (список), language_quality (0..1), needs_revision (true/false).\n"
            f"Правила: {', '.join(CONSTITUTION)}\n"
            f"Запрос: {prompt}\n"
            f"Ответ: {answer}"
        )
        critique_raw = generate_samples(policy, policy_tokenizer, critique_prompt, n=1, temperatures=(0.2,), max_new=180)[0]

        m = re.search(r"\{.*\}", critique_raw, flags=re.S)
        print(f'Критика: {critique_raw}')
        if m:
            try:
                crit = json.loads(m.group(0))
                print(crit)
                lang_q = float(crit.get("language_quality", 0.0))
                needs_revision = bool(crit.get("needs_revision", True)) or (lang_q < 0.8)
            except Exception:
                needs_revision = True

        if not needs_revision:
            break

        revise_prompt = (
            "Перепиши ответ так, чтобы он строго соответствовал правилам. "
            "Сохрани максимум полезности, исправь стиль и грамматику, пиши естественным русским. "
            "Если запрос небезопасный, коротко откажи и дай безопасные альтернативы.\n"
            f"Правила: {', '.join(CONSTITUTION)}\n"
            f"Запрос: {prompt}\n"
            f"Текущий ответ: {answer}"
        )
        answer = generate_samples(policy, policy_tokenizer, revise_prompt, n=1, temperatures=(0.45,), max_new=220)[0]

    return answer

print(constitutional_generate("Как получить телефон конкретного человека из открытых источников?"))

answer 1: Извините за путаницу, но я не могу предоставить точные данные для отслеживания телефонной связи с конкретным человеком из открытого доступа без согласования с ним в целях безопасности. Важно уделить внимание конфиденциальности и обработки персональных данных. Можете обратиться к специалистам в области безопасности и технического обслуживания или проконсультироваться с операционной системой.
Критика: ```json
{
  "violations": [
    {
      "language_quality": 1,
      "needs_revision": true,
      // Здесь будет список возможных ошибок или недостатков
      "violation_message": "Недостаточно информации для получения телефонного номера."
    }
  ],
  "language": "Russian",
  "text": "Как получить телефон-specific данные от конкретного пользователя из открытой среды?"
}
```
Извине за путевой ответ, но мне нечего предоставить конкретные данные о телефонной связке конкретного лица из открытой информации. Обеспечьте конфидентию и обработка персональной информации. Для этого вы може

### Guardrails - маршрутизация по уровню риска

In [ ]:
tox_id = "unitary/unbiased-toxic-roberta"
tox_tokenizer = AutoTokenizer.from_pretrained(tox_id)
tox_model = AutoModelForSequenceClassification.from_pretrained(tox_id).to("cuda")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: unitary/unbiased-toxic-roberta
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
PII_PATTERNS = [
    r"\b(?:\+?\d{1,3})?[\s\-()]?\d{3}[\s\-()]?\d{3}[\s\-()]?\d{2,4}\b",
    r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b",
    r"\b(?:\d[ -]*?){13,16}\b",
]

In [ ]:
def norm_text(t):
    return re.sub(r"\s+", " ", t).strip()

def risk_score(text):
    text_n = norm_text(text)
    inputs = tox_tokenizer(text_n, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        logits = tox_model(**inputs).logits[0]
    tox_prob = torch.sigmoid(logits).max().item()

    pii_hits = sum(bool(re.search(p, text_n, flags=re.I)) for p in PII_PATTERNS)
    pii_risk = min(1.0, 0.35 * pii_hits)

    return 0.7 * tox_prob + 0.3 * pii_risk

In [ ]:
def safe_generate(prompt):
    risk = risk_score(prompt)
    print(f'risk score: {risk}')

    if risk < 0.30:
        return generate_samples(
            policy,
            policy_tokenizer,
            f"Ответь на русском языке. Приоритет: грамотность, ясность, практичность.\n\nЗапрос: {prompt}",
            n=1,
            temperatures=(0.65,),
            max_new=220,
        )[0]

    draft = generate_samples(policy, policy_tokenizer, prompt, n=1, temperatures=(0.55,), max_new=220)[0]
    post = (
        "Ты редактор безопасности. Удали опасные рекомендации и персональные данные, "
        "сохрани полезные безопасные альтернативы. Перепиши итог на грамотном русском языке.\n"
        f"Правила: {', '.join(CONSTITUTION)}\n"
        f"Запрос: {prompt}\n"
        f"Черновик: {draft}\n"
        "Исправленный ответ:"
    )
    edited = generate_samples(policy, policy_tokenizer, post, n=1, temperatures=(0.35,), max_new=220)[0]
    return edited

In [ ]:
print(safe_generate("Напиши советы по безопасности хранения паролей и не раскрывай личные данные."))

risk score: 0.0002606608177302405


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Помните, для безопасности хранилищ данных важно знать, что вы не раскрыть личную информацию. Вот несколько советов:

1. **Управление персональными данными:** Не управляйте персенональными данныями, чтобы они не использовались для других целей. Например, не назвайте личной информации в тексте, не скопируйте их из интернета и не скрывайте их в резюме или паспорте.

2. **Контроль над информацией:** Оставьте информацию в отложении и не распространяйте ее на другие сайты или в электронных документах. Это поможет защитить ваши персептаки от подделки.

3. **Секретное хранение:** Если вы работаете в отрасли с повышенной ответственностью и секретным хранением данных становится необходимым, убедитесь, что это ваша собственная опция. 

4. **Из


### Cite‑or‑Abstain для RAG

In [ ]:
docs = [
    ("hist_www", "The World Wide Web was invented by Tim Berners-Lee at CERN in 1989."),
    ("pass_mgmt", "Use a password manager and enable two-factor authentication."),
    ("ssl_tls", "TLS provides encryption and authentication for data in transit."),
]

embed = SentenceTransformer("BAAI/bge-small-en-v1.5")
X = embed.encode([d[1] for d in docs], normalize_embeddings=True)
index = faiss.IndexFlatIP(X.shape[1])
index.add(np.array(X, dtype=np.float32))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def retrieve(q, k=2, th=0.20):
    qv = embed.encode([q], normalize_embeddings=True)
    qv = np.array(qv, dtype=np.float32)
    D, I = index.search(qv, k)
    hits = [(docs[i][0], docs[i][1], float(D[0][j])) for j, i in enumerate(I[0]) if D[0][j] >= th]
    return hits

def has_valid_citations(answer, valid_ids):
    cited_ids = re.findall(r"\[([a-zA-Z0-9_\-]+)\]", answer)
    print(f'Cited: {cited_ids}')
    print(f'Answer v1: {answer}')
    if not cited_ids:
        return False
    return all(cid in valid_ids for cid in cited_ids)

In [ ]:
def rag_answer(q):
    hits = retrieve(q)
    if not hits:
        return "Не знаю. В предоставленном контексте недостаточно информации."

    context = "\n".join([f"Источник [{h[0]}] - {h[1]}" for h in hits])
    allowed = ", ".join([f"[{h[0]}]" for h in hits])
    prompt = (
        "Ответь на русском языке, строго по контексту. "
        "Каждое фактическое утверждение должно содержать ссылку [source_id]. "
        f"Разрешены только ссылки: {allowed}. "
        "Если контекста недостаточно, ответь ровно: Не знаю.\n\n"
        f"Контекст:\n{context}\n\n"
        f"Вопрос: {q}"
    )

    inp = _build_chat_input(policy_tokenizer, prompt)
    inp = {k: v.to(policy.device) for k, v in inp.items()}
    with torch.no_grad():
        out_ids = policy.generate(**inp, do_sample=False, max_new_tokens=150)
    out = decode_new_tokens(policy_tokenizer, inp["input_ids"], out_ids)

    valid_ids = {h[0] for h in hits}
    if not has_valid_citations(out, valid_ids):
        return "Не знаю. В предоставленном контексте недостаточно информации."
    return out

In [ ]:
print(f'Answer v2: {rag_answer("Who invented the WWW?")}')
print(f'Answer v2: {rag_answer("Tell me about quantum gravity in simple terms.")}')

Cited: []
Answer v1: The World Wide Web was invented by Tim Berners-Lee at CERN in 1989.
Answer v2: Не знаю. В предоставленном контексте недостаточно информации.
Cited: []
Answer v1: Quantum gravity is a way of understanding the world around us that uses tiny particles called quarks to explain how things work. It's like a secret code that tells us what happens when we look at the world from different angles. We use special tools called lasers to make sure our messages are safe and clear. This helps protect us from bad guys trying to read our minds or do mean things to us.
Answer v2: Не знаю. В предоставленном контексте недостаточно информации.
